# 01 — Temperature Interpolation

Three-step workflow:

| Step | Parameter | This notebook |
|------|-----------|---------------|
| 1 | `data=` | Meteostat weather API (or offline synthetic) |
| 2 | `boundary=` | Named place or 4-corner bbox |
| 3 | `method=` | IDW · Kriging · Spline · Natural Neighbor |

**Offline cells** run without any network access.  
**🌐 Network cells** fetch live Meteostat data.  
**🛰 GEE cells** validate against MODIS LST (requires `earthengine authenticate`).

In [ ]:
%pip install -q "geointerpo[kriging,viz,data,raster]"

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import pandas as pd
from geointerpo import Pipeline

## Configuration

In [ ]:
BOUNDARY = (5.0, 44.0, 25.0, 56.0)   # Central Europe  (min_lon, min_lat, max_lon, max_lat)
DATE      = '2024-07-15'
RESOLUTION = 0.25  # degrees (~25 km)

---
## Part A — Offline demo (no network needed)

Uses built-in synthetic station data.  Good for exploring methods and parameters.

In [ ]:
result = Pipeline(
    data='sample',                  # Step 1 — built-in synthetic data
    variable='temperature',
    boundary=BOUNDARY,              # Step 2 — four-corner bbox
    method=['idw', 'kriging', 'spline', 'natural_neighbor'],  # Step 3
    method_params={
        'idw':    {'power': 2},
        'kriging': {'variogram_model': 'spherical'},
        'spline': {'spline_type': 'regularized'},
    },
    resolution=RESOLUTION,
    cv_folds=5,
).run()

In [ ]:
print('Cross-validation metrics:')
result.metrics_table()

In [ ]:
result.plot()
plt.suptitle(f'Temperature — synthetic data (offline demo)', y=1.02)
plt.show()

### Station observations

In [ ]:
ax = result.stations.plot(
    column='value', cmap='RdYlBu_r', legend=True,
    figsize=(9, 6), markersize=40, edgecolor='k', linewidth=0.4
)
ax.set_title('Station observations (synthetic)')
plt.tight_layout()

---
## Part B — 🌐 Live Meteostat data

Fetches real daily weather station data.  Requires network access.

In [ ]:
result_live = Pipeline(
    data='meteostat',              # Step 1 — live Meteostat API
    variable='temperature',
    date=DATE,
    boundary='Central Europe',     # Step 2 — place name via Nominatim
    method=['idw', 'kriging', 'spline', 'natural_neighbor'],  # Step 3
    method_params={
        'kriging': {'variogram_model': 'spherical'},
    },
    resolution=RESOLUTION,
    cv_folds=5,
).run()

print(f'Loaded {len(result_live.stations)} real weather stations')

In [ ]:
result_live.metrics_table()

In [ ]:
result_live.plot()
plt.suptitle(f'Tavg — {DATE}  (Meteostat)', y=1.02)
plt.show()

### Variogram (Kriging)

In [ ]:
from geointerpo import viz
from geointerpo.interpolators import KrigingInterpolator

kriging_model = KrigingInterpolator(variogram_model='spherical').fit(result_live.stations)
fig = viz.plot_variogram(kriging_model, title=f'Spherical variogram — Tavg {DATE}')
plt.show()

### With DEM covariate (elevation)

Adds SRTM elevation as a covariate for ML/Regression-Kriging methods.

In [ ]:
result_dem = Pipeline(
    data='sample',
    variable='temperature',
    boundary=BOUNDARY,
    method=['rf', 'rk'],           # ML methods use DEM as covariate
    include_dem=True,
    dem_source='synthetic',        # use 'gee' or 'srtm' with network
    resolution=RESOLUTION,
    cv_folds=5,
).run()

result_dem.metrics_table()

### Interactive map

In [ ]:
# Interactive map — requires: pip install 'geointerpo[notebooks]'
# (leafmap + localtileserver are notebook-only tools, not part of the core library)
try:
    import leafmap
    import tempfile, os
    da = result.grid
    with tempfile.NamedTemporaryFile(suffix=".tif", delete=False) as f:
        tmp = f.name
    da.rio.set_spatial_dims(x_dim="lon", y_dim="lat").rio.write_crs("EPSG:4326").rio.to_raster(tmp)
    m = leafmap.Map(center=[float(da.lat.mean()), float(da.lon.mean())], zoom=6)
    m.add_raster(tmp, colormap="RdYlBu_r", layer_name=da.name or "interpolated")
    display(m)
except ImportError:
    print("Interactive map requires: pip install 'geointerpo[notebooks]'")
    result.plot()
except Exception as e:
    print(f"Map unavailable ({e}); falling back to static plot")
    result.plot()


---
## Part C — 🛰 GEE validation against MODIS LST

Requires: `earthengine authenticate` run once in terminal.

In [ ]:
try:
    import ee  # noqa: F401
    result_gee = Pipeline(
        data='meteostat',
        variable='temperature',
        date=DATE,
        boundary=BOUNDARY,
        method='kriging',
        method_params={'kriging': {'variogram_model': 'spherical'}},
        resolution=RESOLUTION,
        validate_with_gee=True,        # compare against MODIS LST
    ).run()
    
    print('GEE validation metrics (Kriging vs MODIS LST):')
    for k, v in result_gee.gee_metrics.items():
        if k != 'diff_map':
            print(f'  {k}: {v:.3f}')
except ImportError:
    print("GEE section skipped: install earthengine-api and authenticate first")
except Exception as e:
    print(f"GEE error: {e}")


In [ ]:
fig = viz.plot_diff(
    result_gee.gee_reference,
    result_gee.grid,
    title='Kriging − MODIS LST (°C)'
)
plt.show()

---
## Save outputs

In [ ]:
result.save('outputs/temperature', geotiff=True, plot=True)
print('Saved to outputs/temperature/')